In [1]:
import pandas as pd


In [2]:
# 1. Load Data & Split

df = pd.read_excel("CorporateSalary.xlsx")



In [ ]:
# 2. Split in to Independant And Dependant Variable.

X = df.drop(columns=["Salary"])
y = df["Salary"]


In [ ]:
# 3. Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train_true, y_test_true = train_test_split(X, y, test_size=0.2, random_state=42)



In [ ]:
# 2. Define Groups & Orders
ordinal_cols = ["Education", "Designation"]

education_order = ["Bachelor's", "Master's", "PhD"]
designation_order = [
    "Junior Analyst",
    "Senior Analyst",
    "Lead Consultant",
    "Manager",
    "Senior Manager",
    "Director",
]


ohe_cols = ["Department"]
num_cols = ["TenureYears", "ProjectsCompleted", "PerformanceScore"]




In [ ]:
# 3. Create SINGLE Transformer (Encoding + Scaling in parallel)
# Perform Encoding and scaling

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

OrdinalEncoding_01 = ("ord",OrdinalEncoder(categories=[education_order, designation_order]),ordinal_cols,)

OneHotEncoding_01 =  ("ohe",OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"),ohe_cols,)

Scaling = ("num", StandardScaler(), num_cols)

transform_engine = ColumnTransformer(
            transformers=[OrdinalEncoding_01,OneHotEncoding_01,Scaling,]).set_output(transform="pandas")



In [ ]:
# 4. Fit ONCE on X_train

X_train_transformed = transform_engine.fit_transform(X_train)


In [ ]:
# 5. Export into ONE clean transform.pkl file
import joblib
joblib.dump(transform_engine, "transform.pkl")
print("transform.pkl generated successfully!")


In [ ]:
X_train_transformed


In [ ]:
# -------------------------------------------------------------------
# 4. TRAIN LINEAR REGRESSION MODEL
# -------------------------------------------------------------------
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train_transformed, y_train)

# Save model artifact
joblib.dump(model, "model.pkl")
print("Saved model.pkl successfully!")



,ord__Education,ord__Designation,ohe__Department_Engineering,ohe__Department_HR,ohe__Department_Marketing,ohe__Department_Product,ohe__Department_Sales,num__TenureYears,num__ProjectsCompleted,num__PerformanceScore
83,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.028360,1.792519,0.034785
53,0.0,4.0,0.0,0.0,0.0,1.0,0.0,-1.321436,-1.353704,-0.471182
70,1.0,3.0,0.0,0.0,0.0,1.0,0.0,-1.062965,1.461338,0.793736
45,0.0,2.0,0.0,0.0,0.0,0.0,1.0,-1.034245,0.798975,-0.471182
44,0.0,1.0,0.0,0.0,1.0,0.0,0.0,-0.804493,-0.525750,0.287769
39,2.0,1.0,0.0,1.0,0.0,0.0,0.0,0.717618,0.798975,-1.609607
22,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.205842,-1.353704,-1.483116
80,0.0,4.0,1.0,0.0,0.0,0.0,0.0,-0.775774,0.302203,0.667244
10,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-1.407593,0.798975,0.540752
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.717618,-0.194569,0.793736


In [ ]:
# -------------------------------------------------------------------
# 5. EVALUATE ON TRANSFORMED X_Train
# -------------------------------------------------------------------
y_train_pred = model.predict(X_train_transformed)

y_train_pred


In [ ]:
import joblib

# Load the fitted transformer/scaler
transformer = joblib.load("transform.pkl")

# Transform the test data
X_test_transformed = transformer.transform(X_test)

X_test_transformed


In [ ]:
y_test_pred = model.predict(X_test_transformed)

y_test_pred


In [ ]:
import numpy as np
from sklearn import metrics
y_train_true, y_test_true
mse = metrics.mean_squared_error(y_train_true, y_train_pred)
mse


In [ ]:
mae = metrics.mean_absolute_error(y_train_true, y_train_pred)
mae


In [ ]:
rmse = np.sqrt(mse)
rmse


In [ ]:
r2 = metrics.r2_score(y_train_true, y_train_pred)

r2



In [ ]:
n = len(y_train_pred)  # number of observations
p = len(y_train_pred[0]) if len(y_train_true.shape) > 1 else 1  # number of features
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
adj_r2


In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def func_adjusted_r2(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    r2 = r2_score(y_true, y_pred)

    n = len(y_pred)
    p = y_pred.shape[1] if y_pred.ndim > 1 else 1

    if n - p - 1 <= 0:
        raise ValueError(
            "Degrees of freedom (n - p - 1) must be greater than 0."
        )

    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return adj_r2